# Data Preparation

In [72]:
# Imports

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, vstack
from sklearn.feature_extraction.text import TfidfVectorizer


In [73]:
# Load Data

df = pd.read_csv('../01_data/01_raw_data/customer_original.csv')

df


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [74]:
df.describe()

,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


In [75]:
df.dtypes

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object

In [76]:
# Rename

df = df.rename(columns 
               = {
                   'Customer ID': 'CustomerID',
                   'Price': 'UnitPrice',
                   'Invoice': 'InvoiceNo'
}
)

In [77]:
# New Formats

df['CustomerID'] = df['CustomerID'].astype(str)

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)


In [78]:
# New Variables (Date, Time, Revenue etc.)

# Date & Time
df['invoice_date'] = df['InvoiceDate'].dt.date
df['invoice_time'] = df['InvoiceDate'].dt.time
df['invoice_hour'] = df['InvoiceDate'].dt.hour

df['invoice_year'] = df['InvoiceDate'].dt.year
df['invoice_month'] = df['InvoiceDate'].dt.month
df['invoice_weekday_num'] = df['InvoiceDate'].dt.dayofweek  # Monday=0, Sunday=6
df['invoice_weekday'] = df['InvoiceDate'].dt.day_name()
df['is_weekend'] = df['invoice_weekday_num'].isin([5, 6])
df['invoice_calendarweek'] = df['InvoiceDate'].dt.isocalendar().week   


# Time of Day with 4 Categories
def get_time_of_day(hour):
    if 0 <= hour < 6:
        return "night"
    elif 6 <= hour < 10:
        return "morning"
    elif 10 <= hour < 14:
        return "midday"
    elif 14 <= hour < 18:
        return "afternoon"
    else:
        return "evening"

df['invoice_time_5cat'] = df['invoice_hour'].apply(get_time_of_day)


# Revenue 
df['RevenueLine'] = df['Quantity']*df['UnitPrice']

df[
    [
        'Quantity', 
        'UnitPrice',
        'RevenueLine',
    ]
].head()

df[
    [
        'InvoiceDate',
        'invoice_date',
        'invoice_calendarweek',
        'invoice_weekday',
        'is_weekend',
        'invoice_time',
        'invoice_hour',
        'invoice_time_5cat',
        'Quantity',
        'UnitPrice',
        'RevenueLine',
    ]
].head()

,InvoiceDate,invoice_date,invoice_calendarweek,invoice_weekday,is_weekend,invoice_time,invoice_hour,invoice_time_5cat,Quantity,UnitPrice,RevenueLine
0,2009-12-01 07:45:00,2009-12-01,49,Tuesday,False,07:45:00,7,morning,12,6.95,83.4
1,2009-12-01 07:45:00,2009-12-01,49,Tuesday,False,07:45:00,7,morning,12,6.75,81.0
2,2009-12-01 07:45:00,2009-12-01,49,Tuesday,False,07:45:00,7,morning,12,6.75,81.0
3,2009-12-01 07:45:00,2009-12-01,49,Tuesday,False,07:45:00,7,morning,48,2.10,100.8
4,2009-12-01 07:45:00,2009-12-01,49,Tuesday,False,07:45:00,7,morning,24,1.25,30.0


### Transactions Filtering - Purchases, Returns, Cancellations

In [79]:
# Helper variables
df["is_cancellation_invoice"] = df["InvoiceNo"].str.startswith(
    "C",
    na=False
)
df["is_return_quantity"] = df["Quantity"].lt(0)
df["is_non_positive_price"] = df["UnitPrice"].le(0)


# df Purchases
purchases = df[
    df["CustomerID"].notna()
    & df["InvoiceDate"].notna()
    & df["Description"].notna()
    & df["Quantity"].gt(0)
    & df["UnitPrice"].gt(0)
    & ~df["is_cancellation_invoice"]
].copy()

print(purchases.shape)
purchases.head()

(805549, 22)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,invoice_date,invoice_time,...,invoice_month,invoice_weekday_num,invoice_weekday,is_weekend,invoice_calendarweek,invoice_time_5cat,RevenueLine,is_cancellation_invoice,is_return_quantity,is_non_positive_price
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-12-01,07:45:00,...,12,1,Tuesday,False,49,morning,83.4,False,False,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12-01,07:45:00,...,12,1,Tuesday,False,49,morning,81.0,False,False,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12-01,07:45:00,...,12,1,Tuesday,False,49,morning,81.0,False,False,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-12-01,07:45:00,...,12,1,Tuesday,False,49,morning,100.8,False,False,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-12-01,07:45:00,...,12,1,Tuesday,False,49,morning,30.0,False,False,False


In [80]:
purchases = df[
    df['CustomerID'].notna()
    & df['InvoiceDate'].notna()
    & df['Description'].notna()
    & df['Quantity'].gt(0)
    & df['UnitPrice'].gt(0)
    & ~df['InvoiceNo'].astype(str).str.startswith("C", na=False)
].copy()

# purchases.describe()

In [81]:
# df Returns

returns = df[
    df['CustomerID'].notna()
    & (
        df['is_cancellation_invoice']
        | df['is_return_quantity']
    )
].copy()

returns['return_value'] = (
    returns['Quantity'] * returns['UnitPrice']
)

returns.shape

(18744, 23)

In [82]:
pd.crosstab(
    df['is_cancellation_invoice'],
    df['is_return_quantity'],
    margins=True
)

is_return_quantity,False,True,All
is_cancellation_invoice,,,
False,1044420,3457,1047877
True,1,19493,19494
All,1044421,22950,1067371


In [83]:
# df Cancellations

negative_prices = df[df['UnitPrice'] < 0].copy()

negative_prices[
    [
        'InvoiceNo',
        'StockCode',
        'Description',
        'Quantity',
        'UnitPrice',
        'CustomerID',
    ]
].head(20)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID
179403,A506401,B,Adjust bad debt,1,-53594.36,NaN
276274,A516228,B,Adjust bad debt,1,-44031.79,NaN
403472,A528059,B,Adjust bad debt,1,-38925.87,NaN
825444,A563186,B,Adjust bad debt,1,-11062.06,NaN
825445,A563187,B,Adjust bad debt,1,-11062.06,NaN


In [84]:
# Descriptions

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.90,
    max_features=100
)

X_text_line = vectorizer.fit_transform(
    purchases['Description']
)

customer_ids = purchases['CustomerID'].to_numpy()
unique_customer_ids = np.sort(np.unique(customer_ids))

customer_text_vectors = []

for customer_id in unique_customer_ids:
    customer_mask = customer_ids == customer_id
    customer_vector = X_text_line[customer_mask].mean(axis=0)

    customer_text_vectors.append(csr_matrix(customer_vector))

X_text_customer = vstack(customer_text_vectors)

In [85]:
print("Number Customers:", purchases.groupby('CustomerID').count().shape)
print("Shape :", X_text_customer.shape)


Number Customers: (5878, 21)
Shape : (5878, 100)


In [86]:
feature_names = vectorizer.get_feature_names_out()

print(feature_names)
print(f"Number Text Features: {len(feature_names)}")

['12' '20' '60' 'antique' 'assorted' 'bag' 'bird' 'birthday' 'black'
 'blue' 'bottle' 'bowl' 'box' 'bunting' 'cake' 'cake cases' 'candle'
 'candles' 'card' 'cases' 'ceramic' 'christmas' 'clock' 'colour' 'cream'
 'cutlery' 'decoration' 'design' 'dolly' 'door' 'doormat' 'fairy'
 'fairy cake' 'feltcraft' 'flower' 'frame' 'garden' 'girl' 'glass' 'green'
 'hanging' 'hanging heart' 'heart' 'hearts' 'holder' 'home' 'hot'
 'hot water' 'ivory' 'jumbo' 'jumbo bag' 'kit' 'large' 'light'
 'light holder' 'lights' 'love' 'lunch' 'lunch bag' 'metal' 'metal sign'
 'mini' 'mug' 'pack' 'paisley' 'paper' 'party' 'pink' 'polkadot' 'red'
 'red retrospot' 'red spotty' 'regency' 'retro' 'retrospot' 'rose' 'set'
 'sign' 'silver' 'skull' 'small' 'spaceboy' 'spot' 'spotty' 'star'
 'strawberry' 'tea' 'tin' 'trinket' 'union' 'vintage' 'water'
 'water bottle' 'white' 'wicker' 'wood' 'wooden' 'woodland' 'wrap' 'zinc']
Number Text Features: 100


In [87]:
text_feature_map = pd.DataFrame({
    "column_index": range(len(feature_names)),
    "term": feature_names
})

text_feature_map.head(20)

,column_index,term
0,0,12
1,1,20
2,2,60
3,3,antique
4,4,assorted
5,5,bag
6,6,bird
7,7,birthday
8,8,black
9,9,blue


In [88]:
text_feature_map.to_csv(
    "../03_results/tfidf_feature_vocabulary.csv",
    index=False
)

In [89]:
df.to_csv("../01_data/02_processed_data/customer_processed.csv", index=False)